In [2]:
import numpy as np
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
from pynq import Overlay
from pynq import allocate
import time

print("--- STAP 1: Hardware Overlay Laden ---")
overlay = Overlay("./pooling.bit")

# Dynamische detectie van de IP-core naam
ip_name = list(overlay.ip_dict.keys())[0]
pooling_ip = getattr(overlay, ip_name)
print(f"Overlay succesvol geladen! IP-core gedetecteerd als: {ip_name}")

# --- CONFIGURATIE ---
# Definieer 10 vaste URL's met afbeeldingen (geschaald naar 256x256 via Unsplash API)
afbeeldings_urls = [
    "https://images.unsplash.com/photo-1542291026-7eec264c27ff?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1523275335684-37898b6baf30?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1505740420928-5e560c06d30e?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1572635196237-14b3f281503f?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1560343090-f0409e92791a?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1491553895911-0055eca6402d?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1526170375885-4d8ecf77b99f?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1546868871-7041f2a55e12?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1583394838336-acd977736f90?auto=format&fit=crop&w=256&h=256",
    "https://images.unsplash.com/photo-1503602642458-232111445657?auto=format&fit=crop&w=256&h=256"
]

IN_WIDTH, IN_HEIGHT = 256, 256
OUT_WIDTH, OUT_HEIGHT = 128, 128  # Exact de helft door 2x2 pooling
POOL_MODE = 0  # 0 = Max Pooling, 1 = Average Pooling, 2 = Min Pooling

print("\n--- STAP 2: Fysiek Geheugen Alloceren (Contiguous Memory) ---")
in_buffer = allocate(shape=(IN_HEIGHT, IN_WIDTH), dtype=np.uint32)
out_buffer = allocate(shape=(OUT_HEIGHT, OUT_WIDTH), dtype=np.uint32)

print(f"Invoerbuffer adres: {hex(in_buffer.device_address)}")
print(f"Uitvoerbuffer adres: {hex(out_buffer.device_address)}")

headers = {"User-Agent": "Mozilla/5.0"}

# --- BATCH VERWERKINGS-LUS ---
for idx, url in enumerate(afbeeldings_urls):
    print(f"\n================ OPHALEN & VERWERKEN: AFBEELDING {idx+1}/10 ================")
    try:
        # 1. Download de afbeelding via de URL
        response = requests.get(url, headers=headers, timeout=5)
        img = Image.open(BytesIO(response.content)).convert("RGBA")
        
        # Zorg dat de afmetingen exact kloppen met de hardware-verwachting
        img = img.resize((IN_WIDTH, IN_HEIGHT))
        img_np = np.array(img, dtype=np.uint32)
        
        # 2. Pak de RGBA-bytes in naar 32-bit datawords (0xAA_BB_GG_RR)
        ingepakte_pixels = (img_np[:,:,3] << 24) | (img_np[:,:,2] << 16) | (img_np[:,:,1] << 8) | img_np[:,:,0]
        
        # Kopieer naar de fysieke invoerbuffer en flush naar het DDR-geheugen
        in_buffer[:] = ingepakte_pixels
        in_buffer.flush()
        
        # 3. Configureer de FPGA registers
        try:
            pooling_ip.register_map.breedte = IN_WIDTH
            pooling_ip.register_map.hoogte = IN_HEIGHT
            pooling_ip.register_map.mode = POOL_MODE
        except AttributeError:
            pass # Soms zitten deze in de CONTROL_BUS offset, dat vangt pynq intern op
            
        # Adreskoppeling (vangt zowel 64-bit split als directe 32-bit registers op)
        try:
            pooling_ip.register_map.invoer_pixels_1 = in_buffer.device_address & 0xFFFFFFFF
            pooling_ip.register_map.invoer_pixels_2 = (in_buffer.device_address >> 32) & 0xFFFFFFFF
            pooling_ip.register_map.uitvoer_pixels_1 = out_buffer.device_address & 0xFFFFFFFF
            pooling_ip.register_map.uitvoer_pixels_2 = (out_buffer.device_address >> 32) & 0xFFFFFFFF
        except AttributeError:
            pooling_ip.register_map.invoer_pixels = in_buffer.device_address
            pooling_ip.register_map.uitvoer_pixels = out_buffer.device_address

        # 4. Start de hardware core
        t_start = time.time()
        pooling_ip.write(0x00, 1) # Forceer AP_START
        
        # Wacht tot de core klaar is (AP_DONE bit 1) of time-out optreedt
        timeout = 2.0
        success = False
        while (time.time() - t_start) < timeout:
            status = pooling_ip.read(0x00)
            if status & 0x02:
                success = True
                break
                
        t_end = time.time()
        
        if not success:
            print(f"❌ Hardware hangt bij afbeelding {idx+1}. Status register 0x00: {bin(status)}")
            continue
            
        print(f"🎉 FPGA verwerking succesvol afgerond in: {(t_end - t_start)*1000:.3f} ms!")
        
        # 5. Cache legen en data teruglezen
        out_buffer.invalidate()
        raw_output = np.array(out_buffer, dtype=np.uint32)
        
        # Pak de 32-bit words weer uit naar een toonbare RGB-matrix
        output_rgb = np.zeros((OUT_HEIGHT, OUT_WIDTH, 3), dtype=np.uint8)
        output_rgb[:,:,0] = (raw_output & 0xFF)         # R
        output_rgb[:,:,1] = ((raw_output >> 8) & 0xFF)  # G
        output_rgb[:,:,2] = ((raw_output >> 16) & 0xFF) # B
        
        # Sla het resultaat lokaal op op de Pynq-Z2
        Image.fromarray(output_rgb).save(f"./gepoold_resultaat_{idx+1}.png")
        
        # 6. Toon de resultaten live in je Jupyter Notebook
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img.convert("RGB"))
        axes[0].axis('off')
        axes[0].set_title(f"Origineel {idx+1} ({IN_WIDTH}x{IN_HEIGHT})")
        
        mode_naam = "Max" if POOL_MODE == 0 else "Average" if POOL_MODE == 1 else "Min"
        axes[1].imshow(output_rgb)
        axes[1].axis('off')
        axes[1].set_title(f"FPGA {mode_naam} Pooling ({OUT_WIDTH}x{OUT_HEIGHT})")
        plt.show()

    except Exception as e:
        print(f"Fout opgetreden bij verwerken van URL: {e}")

# Schoon het fysieke geheugen netjes op na de batch
print("\n--- STAP 4: Geheugen Vrijmaken ---")
del in_buffer
del out_buffer
print("Buffers succesvol opgeruimd. Project voltooid!")

--- STAP 1: Hardware Overlay Laden ---
Overlay succesvol geladen! IP-core gedetecteerd als: apply_hardware_pooli_0

--- STAP 2: Fysiek Geheugen Alloceren (Contiguous Memory) ---
Invoerbuffer adres: 0x15880000
Uitvoerbuffer adres: 0x15860000

================ OPHALEN & VERWERKEN: AFBEELDING 1/10 ================
❌ Hardware hangt bij afbeelding 1. Status register 0x00: 0b0

================ OPHALEN & VERWERKEN: AFBEELDING 2/10 ================
❌ Hardware hangt bij afbeelding 2. Status register 0x00: 0b0

================ OPHALEN & VERWERKEN: AFBEELDING 3/10 ================
❌ Hardware hangt bij afbeelding 3. Status register 0x00: 0b0

================ OPHALEN & VERWERKEN: AFBEELDING 4/10 ================
❌ Hardware hangt bij afbeelding 4. Status register 0x00: 0b0

================ OPHALEN & VERWERKEN: AFBEELDING 5/10 ================
❌ Hardware hangt bij afbeelding 5. Status register 0x00: 0b0

================ OPHALEN & VERWERKEN: AFBEELDING 6/10 ================
❌ Hardware hangt bij a